## Final Project for Data 605 - Big data systems
- Name: Mohammed Ateeq Ur Rehman
- UID: 120872334

In [1]:
# Run this if libraires are not already installed
%pip install petastorm pyarrow pandas matplotlib torch torchvision tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import pandas as pd
import pyarrow.parquet as pq
import os
from pyarrow import fs
import petastorm_bitcoin_processing_utils as utils

##  Part 1: Data Ingestion

### Task: Get the data using coingecko API

In [3]:
output_dir = 'test_bitcoin_data'
os.makedirs(output_dir, exist_ok=True)

base_url = "https://api.coingecko.com/api/v3"
historical_data = []

In [9]:
# Parameters
interval_sec = 30  # every 30 seconds
duration_min = 2   # run for 2 minutes
save_interval = 2  # save every 2 fetches

end_time = time.time() + duration_min * 60
fetch_count = 0
historical_data = []

output_file = os.path.join(output_dir, "bitcoin_price_data.csv")

while time.time() < end_time:
    try:
        price_data = utils.fetch_current_price(base_url)
        historical_data.append(price_data)
        fetch_count += 1
        print(f"Collected data at {price_data['timestamp']}")

        # Save to file every `save_interval` fetches
        if fetch_count % save_interval == 0:
            new_df = pd.DataFrame(historical_data)
            historical_data = []  # clear buffer after saving

            # Append to existing file or create new one
            if os.path.exists(output_file):
                new_df.to_csv(output_file, mode='a', header=False, index=False)
            else:
                new_df.to_csv(output_file, mode='w', header=True, index=False)

    except Exception as e:
        print(f"Error fetching data: {e}")
        break  # Use break instead of exit in notebooks/scripts

    time.sleep(interval_sec)
print("Data collection complete.")

Collected data at 2025-05-18T00:42:17.793458
Collected data at 2025-05-18T00:42:47.901688
Collected data at 2025-05-18T00:43:18.004162
Collected data at 2025-05-18T00:43:48.112282
Data collection complete.


## Part2: Batch Data Storage (Parquet Format)

### Task: Convert the CSV to parquet using PETASTROM

In [12]:
# 1. Load all price CSVs
csv_folder = r"test_bitcoin_data"
df = utils.load_all_csvs_from_folder(csv_folder)
print(f"Loaded {len(df)} rows from CSV files.")

# 2. Save to Parquet
parquet_path = "file:///test_bitcoin_data/parquet"
utils.save_to_parquet_arrow(df, parquet_path)

# 3. Load from Parquet and preview
batches = list(utils.load_from_parquet(input_dir='test_bitcoin_data\\parquet'))

Combined 2 CSV files.
Loaded 388 rows from CSV files.
======== Parquet file written to 
 e:\UMD\Data 605 - PCS2\Project\tutorials1\DATA605\Spring2025\projects\TutorTask188_Spring2025_Batch_Processing_of_Bitcoin_Price_Data_with_Petastorm​\docker_data605_style/test_bitcoin_data/parquet ==========
Loading data from file://e:/UMD/Data 605 - PCS2/Project/tutorials1/DATA605/Spring2025/projects/TutorTask188_Spring2025_Batch_Processing_of_Bitcoin_Price_Data_with_Petastorm​/docker_data605_style/test_bitcoin_data/parquet...


c:\Users\ateeq\anaconda3\Lib\site-packages\petastorm\fs_utils.py:88: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalFileSystem instead.
  self._filesystem = pyarrow.localfs
c:\Users\ateeq\anaconda3\Lib\site-packages\petastorm\etl\dataset_metadata.py:402: FutureWarning: Passing 'use_legacy_dataset=True' to get the legacy behaviour is deprecated as of pyarrow 11.0.0, and the legacy implementation will be removed in a future version. The legacy behaviour was still chosen because a deprecated 'pyarrow.filesystem' filesystem was specified (use the filesystems from pyarrow.fs instead).
  dataset = pq.ParquetDataset(path_or_paths, filesystem=fs, validate_schema=False, metadata_nthreads=10)
c:\Users\ateeq\anaconda3\Lib\site-packages\petastorm\etl\dataset_metadata.py:402: FutureWarning: Specifying the 'metadata_nthreads' argument is deprecated as of pyarrow 8.0.0, and the argument will be removed in a future version
  dataset = pq.ParquetDataset(path_or_pat

Columns of the parquet file

In [13]:
parquet_path = "test_bitcoin_data/parquet/data.parquet"
table = pq.read_table(parquet_path)
schema_columns = table.column_names
print("Columns:", table.column_names)

Columns: ['timestamp', 'price_usd', 'market_cap', 'price_change_24h']


Sample batch data from parquet file

In [14]:
for batch in batches[:1]:
    df = pd.DataFrame(batch)
    print("Batch shape:", df.shape)
    print(df.head())

Batch shape: (4, 388)
                          0                           1    \
0  2025-05-17T18:47:14.689778  2025-05-17T18:47:44.810582   
1                    103057.0                    103057.0   
2        2047317305796.520752        2047317305796.520752   
3                   -0.394764                   -0.394764   

                          2                           3    \
0  2025-05-17T19:05:24.408553  2025-05-17T19:05:54.518471   
1                    103054.0                    103054.0   
2        2047160205666.818115        2047160205666.818115   
3                   -0.445364                   -0.445364   

                          4                           5    \
0  2025-05-17T21:49:08.839872  2025-05-17T21:49:38.944797   
1                    103268.0                    103268.0   
2        2051453556092.881104        2051453556092.881104   
3                    0.438986                    0.438986   

                          6                           7    \